In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


# 第 19 章 · FSDP2：每卡只存一片，用到时再拼

本章回答：DDP 每卡存一整份模型，显存哪里浪费了？ZeRO 1 / 2 / 3 分别切掉了什么、各省多少？FSDP2 的 `fully_shard` 在前向反向里什么时候收集参数、什么时候丢？`reshard_after_forward`、混合精度策略、HSDP 怎么选？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/19-fsdp2.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §ZeRO（`ultrascale-playbook.md` 第 794–1031 行：Memory usage revisited 第 826 行、显存公式第 840–852 行、ZeRO-1 第 857 行、ZeRO-2 第 921 行、ZeRO-3 第 953 行）；官方文档 `torch.distributed.fsdp.fully_shard`（2.14 版已核对，〔在线〕）；多机报告 §2.1；本章输出在本机 Gloo 2 进程上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard
import torch
import torch.nn as nn

import ch19_fsdp2 as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一、二步 显存账 | 16Ψ 与三级 ZeRO | `zero_memory_gb(7e9, 8)` | 112 / 38.5 / 26.25 / 14 GB |
| 第三步 fully_shard | Shard(0) 的 DTensor | `shard_demo` | 全局 (6, 4)，每卡 (3, 4) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `shard_demo`**（来自 `ch19_fsdp2.py`）：下面要用到的小工具。

In [ ]:
def shard_demo(rank, world):
    mesh = init_device_mesh("cpu", (world,))
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(4, 6), nn.Linear(6, 2))
    for layer in m:
        fully_shard(layer, mesh=mesh)
    fully_shard(m, mesh=mesh)
    p = m[0].weight
    m(torch.ones(1, 4)).sum().backward()
    return dict(type=type(p).__name__, global_shape=tuple(p.shape), local_shape=tuple(p.to_local().shape),
                placements=str(p.placements), grad_local=tuple(m[0].weight.grad.to_local().shape))

**函数 `zero_memory_gb`**（来自 `ch19_fsdp2.py`）：混合精度 AdamW 每卡显存（GB）：DDP 16Ψ，ZeRO-1/2/3（Playbook 第 840–873 行公式）。

In [ ]:
def zero_memory_gb(psi: float, n: int) -> dict:
    """混合精度 AdamW 每卡显存（GB）：DDP 16Ψ，ZeRO-1/2/3（Playbook 第 840–873 行公式）。"""
    g = 1e9
    return dict(DDP=16 * psi / g, ZeRO1=(4 * psi + 12 * psi / n) / g, ZeRO2=(2 * psi + 14 * psi / n) / g,
                ZeRO3=16 * psi / n / g)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一、二步 7B、8 卡每卡显存（GB）：", {k: round(v, 2) for k, v in zero_memory_gb(7e9, 8).items()})

In [ ]:
print("\n第三步 fully_shard 之后：")
for r, res in enumerate(run(M.shard_demo)):
    print(f"  rank {r}: {res}")

## 3. 代码里才看得到的

1. **FSDP2 在 CPU + Gloo 上也能跑**：所以本机能看到参数被切开的样子，只是没有加速。
2. **梯度也是 DTensor**：`grad.to_local()` 也是 (3, 4)，每卡只存自己那片的梯度。

## 4. 自己改一改

- 把 `nn.Linear(4, 6)` 改成 `nn.Linear(4, 5)`，看不能整除时每卡分到几行。
- `zero_memory_gb(70e9, 64)` 算 70B 64 卡。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 混合精度每参十六字节，DDP 人人一整套；
> ZeRO 一切优化器、二再切梯度、三连参数也切，三级每卡十六 Ψ 除以 N；
> 层前拼、用完丢、反向后梯度 reduce_scatter；从下往上逐层套，机内切、机间复制叫 HSDP。